# 04 — Disease-relevant regulons

Finds regulons that are relevant to the disease: survival-associated (Cox) **or** hallmark-enriched (GO + Lin similarity), **and** co-expression-validated in an independent cohort.
`{prefix}` = `CANCER.lower()` (e.g. `brca`), `{val}` = `VAL_NAME.lower()` (e.g. `metabric`); paths are set in the config cell.

## Inputs

| File | From | Format |
|---|---|---|
| `{prefix}_output/eigengenes.csv` | 02 | regulons × patients; index = regulon ID (int), columns = TCGA patient IDs; values = PC1 activity score |
| `{prefix}_output/clinical_data.csv` | 03 | index = `PATIENT_ID`; columns `OS_STATUS` (0/1), `OS_MONTHS` (float) |
| `{prefix}_output/regulons.json` | 02 | `{regulon ID (str): [member genes, Ensembl IDs]}` |
| `{prefix}_zscored.csv` (in `base_dir`) | 01 | genes (symbols) × patients, z-scored; only the gene list is used (GO background) |
| `GO_DIR/go-basic.obo` | GO website | GO term hierarchy (DAG) |
| `GO_DIR/goa_human.gaf.gz` | GO website (downloaded if missing) | human gene → GO term annotations |
| `miner/data/identifier_mappings.txt` | miner package | Ensembl ↔ gene symbol mapping |
| `val_dir/data_mrna*.txt` | validation cohort (cBioPortal) | rows = genes (`Hugo_Symbol`, `Entrez_Gene_Id`), columns = samples; log2 expression |
| `val_dir/data_clinical_patient.txt` | validation cohort (cBioPortal) | `#` header lines, then `PATIENT_ID`, `OS_MONTHS`, `OS_STATUS` (`"1:DECEASED"`), optional `VITAL_STATUS` |

## Outputs (all in `{prefix}_output/` unless noted)

| File | Format |
|---|---|
| `regulon_cox_results.csv` | index = regulon ID; `HR`, `p_value`, `disease_relevant` (True if p ≤ 0.05) |
| `disease_relevant_regulons_survival_only.csv` | no header; one regulon ID per line (Cox p ≤ 0.05) |
| `GO_DIR/goa_human.gaf` | unzipped GO annotation file (written in the GO_DIR folder) |
| `go_bio_symbol.pkl` (+ `_checkpoint`) | dict `{regulon ID (str): [significant GO BP term IDs]}` (FDR ≤ 0.05); cached, ~47 min to compute |
| `lin_cache.pkl` (+ `_checkpoint`) | dict `{regulon GO term: {hallmark GO term: Lin similarity 0–1}}`; cached, ~15 min to compute |
| `disease_relevant_regulons_survival_or_hallmark.csv` | no header; regulon IDs that are survival-significant OR hallmark-enriched |
| `{val}_zscored.pkl` | pickled DataFrame, genes (symbols) × validation samples, z-scored per gene; used by 05 |
| `{val}_clinical.csv` | index = `PATIENT_ID`; `OS_MONTHS`, `OS_STATUS` (0/1), `DSS_STATUS` (0/1/NaN, only if the cohort records cause of death); used by 05 |
| `{val}_random_regulon_variance.csv` | null distribution; index = gene-set of size N; columns `N`, `Variance` (variance of all regulon genes averaged per patient), `Variance_explained` (PC1); 250 rows per size; cached |
| `{val}_regulon_coherence.csv` | one row per testable regulon: `N`, `Variance`, `Variance_expained`, `Modules` (row position), `Regulon` (ID), `Perm_pass` (0/1), `PC1_pass` (0/1) |
| `{val}_coexp_validated_regulons.csv` | no header; regulon IDs passing permutation p ≤ 0.05 AND PC1 ≥ 0.3 |
| `final_disease_relevant_regulons.csv` | no header; **final set** = (survival OR hallmark) AND co-expression validated. Written twice: first as the union (cell 20), then overwritten by the final set (last cell). Always read with `header=None` |


**No independent cohort:** if the cancer is not in `VALIDATION_COHORTS`, the co-expression validation cells are skipped, the final set is (survival OR hallmark) only, and `validation_status.txt` says the results are not externally validated.

> **GBM note.** GBM expression comes from `01_gbm_expression` (RNA-seq and microarray patients, z-scored within platform), not from recount3. The independent cohort is Gravendeel et al. 2009 (GSE16011), which contains gliomas of all grades, not only GBM.

In [26]:
# Necessary imports and path specifications
import pandas as pd
import numpy as np
import os
import pickle
import warnings
import gzip
import json
import requests
import time
import miner.miner as miner
import glob, io, contextlib

from miner.miner import *
from collections import defaultdict, Counter
from goatools.anno.gaf_reader import GafReader
from lifelines import CoxPHFitter
from goatools.obo_parser import GODag
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from goatools.go_enrichment import GOEnrichmentStudy
from goatools.semantic import lin_sim, TermCounts

warnings.filterwarnings('ignore')

# Repository root: holds config.py (paths and per-cancer settings) and driver_tiers.py
import sys
repo_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
import config

# CANCER CONFIGURATION: the cancer and the data root come from config.py
CANCER     = config.CANCER
base_dir   = config.BASE_DIR

# this cancer's own folder: holds <prefix>_data, <prefix>_notebooks and <prefix>_output
cancer_dir = os.path.join(base_dir, CANCER.lower())

# this cancer's expression / mutation files (written by 01 and 02)
data_dir   = os.path.join(cancer_dir, f"{CANCER.lower()}_data")
prefix     = CANCER.lower()
output_dir = os.path.join(cancer_dir, f"{prefix}_output")

# GO files (go-basic.obo, goa_human.gaf) are shared across cancers
GO_DIR = config.GO_DIR

# Independent cohort used for co-expression validation: cancer -> (name, folder inside base_dir)
VALIDATION_COHORTS = config.VALIDATION_COHORTS
# A cancer that is not listed here has no independent cohort: the co-expression validation is skipped and the
# results are marked "not externally validated" (validation_status.txt in the output folder)
VAL_NAME, val_folder = VALIDATION_COHORTS.get(CANCER, (None, None))
HAS_VALIDATION = VAL_NAME is not None

# file prefix: "VAL_NAME" -> same file names as before
val     = VAL_NAME.lower() if HAS_VALIDATION else "none"
val_dir = os.path.join(data_dir, val_folder) if HAS_VALIDATION else None

# CPU workers for the null-distribution step
N_JOBS = 12

# Other necessary paths
miner_dir  = os.path.dirname(miner.__file__)
idmap_path = os.path.join(miner_dir, "data", "identifier_mappings.txt")
godag = GODag(os.path.join(GO_DIR, "go-basic.obo"))
print(f"Cancer: {CANCER} | validation cohort: {VAL_NAME if HAS_VALIDATION else 'none (not externally validated)'} | GO DAG: {len(godag)} terms")

go-basic.obo: fmt(1.2) rel(2026-03-25) 41,853 Terms
Cancer: GBM | validation cohort: GRAVENDEEL | GO DAG: 41853 terms


In [2]:
# Load regulon eigengenes: PC1 activity score per regulon per patient.
# Produced already in 02_regulons_clusters_states.ipynb.
eigengenes = pd.read_csv(os.path.join(output_dir, "eigengenes.csv"), index_col=0)
eigengenes.columns = eigengenes.columns.astype(str)

print(f"eigengenes shape: {eigengenes.shape}")
print(f"regulon id sample: {eigengenes.index[:5].tolist()}")
print(f"patient id sample: {eigengenes.columns[:3].tolist()}")

eigengenes shape: (3961, 550)
regulon id sample: [0, 1, 2, 3, 4]
patient id sample: ['TCGA-02-0001', 'TCGA-02-0004', 'TCGA-02-0007']


In [3]:
# Load clinical data (survival status and time) saved by 03
clinical = pd.read_csv(os.path.join(output_dir, "clinical_data.csv"), index_col=0)
print(f"Loaded clinical_data.csv from 03: {clinical.shape}")

Loaded clinical_data.csv from 03: (594, 2)


In [4]:
# Align to patients present in both eigengenes and clinical data.
common_patients  = list(set(eigengenes.columns) & set(clinical.index))
eigen_aligned    = eigengenes.loc[:, common_patients].T   # patients x regulons
clinical_aligned = clinical.loc[common_patients, ["OS_MONTHS", "OS_STATUS"]]

print(f"eigengenes patients: {eigengenes.shape[1]}")
print(f"clinical patients:   {clinical.shape[0]}")
print(f"common patients:     {len(common_patients)}")

eigengenes patients: 550
clinical patients:   594
common patients:     545


In [5]:
# Sanity check on a single regulon before batch fitting
# CoxPHFitter regresses time-to-event (OS_MONTHS) against the regulon's eigengene value,
# censoring on OS_STATUS (1 = death observed, 0 = censored/alive at last follow-up).
# HR > 1 means higher eigengene activity associates with worse (shorter) survival.
test_regulon = eigen_aligned.columns[0]

df_test = clinical_aligned.copy()
df_test["eigengene"] = eigen_aligned[test_regulon].values
df_test = df_test.dropna()

cph = CoxPHFitter()
cph.fit(df_test, duration_col="OS_MONTHS", event_col="OS_STATUS")
print(f"Regulon {test_regulon}: HR={cph.hazard_ratios_['eigengene']:.4f}, p={cph.summary.loc['eigengene','p']:.4f}")

Regulon 0: HR=0.9740, p=0.6217


In [6]:
# Batch Cox test across every regulon, 500 at a time 
cox_results = []
regulon_list = eigen_aligned.columns.tolist()
batch_size = 500

for batch_start in range(0, len(regulon_list), batch_size):
    batch = regulon_list[batch_start:batch_start + batch_size]
    for regulon in batch:
        df_cox = clinical_aligned.copy()
        df_cox["eigengene"] = eigen_aligned[regulon].values
        df_cox = df_cox.dropna()
        try:
            cph = CoxPHFitter()
            cph.fit(df_cox, duration_col="OS_MONTHS", event_col="OS_STATUS")
            hr = cph.hazard_ratios_["eigengene"]
            p  = cph.summary.loc["eigengene", "p"]
            cox_results.append({"regulon": regulon, "HR": hr, "p_value": p})
        except Exception:
            cox_results.append({"regulon": regulon, "HR": np.nan, "p_value": np.nan})
    print(f"completed {min(batch_start + batch_size, len(regulon_list))}/{len(regulon_list)} regulons")

cox_df = pd.DataFrame(cox_results).set_index("regulon")

completed 500/3961 regulons
completed 1000/3961 regulons
completed 1500/3961 regulons
completed 2000/3961 regulons
completed 2500/3961 regulons
completed 3000/3961 regulons
completed 3500/3961 regulons
completed 3961/3961 regulons


In [7]:
# Regulons here are disease-relevant but only based on survival. Have not been checked against other criteria
# Filter to p-val threshold

cox_df["disease_relevant"] = cox_df["p_value"] <= 0.05
cox_df.to_csv(os.path.join(output_dir, "regulon_cox_results.csv"))

# Save outputs
disease_relevant = cox_df[cox_df["disease_relevant"]].index.tolist()
pd.Series(disease_relevant).to_csv(
    os.path.join(output_dir, "disease_relevant_regulons_survival_only.csv"),
    index=False, header=False
)

print(f"total regulons tested: {len(cox_df)}")
print(f"survival-significant (p<=0.05): {len(disease_relevant)}")
print(f"high risk (HR>1, p<=0.05): {((cox_df['HR']>1) & cox_df['disease_relevant']).sum()}")
print(f"low risk  (HR<1, p<=0.05): {((cox_df['HR']<1) & cox_df['disease_relevant']).sum()}")

total regulons tested: 3961
survival-significant (p<=0.05): 1354
high risk (HR>1, p<=0.05): 995
low risk  (HR<1, p<=0.05): 359


In [8]:
# Download the GO annotation file (human gene -> GO term) if it isn't already in GO_DIR
gbm_dir = GO_DIR

gaf_path = os.path.join(gbm_dir, "goa_human.gaf.gz")
if not os.path.exists(gaf_path):
    resp = requests.get(
        "https://current.geneontology.org/annotations/goa_human.gaf.gz",
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=60,
    )
    resp.raise_for_status()
    with open(gaf_path, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded {len(resp.content)/1e6:.1f} MB")
else:
    print("Reusing existing goa_human.gaf.gz")

Reusing existing goa_human.gaf.gz


In [9]:
# Unzip the annotation file (goatools reads the plain .gaf)
gaf_plain_path = os.path.join(gbm_dir, "goa_human.gaf")

with gzip.open(gaf_path, "rt", encoding="utf-8", errors="replace") as fin, \
     open(gaf_plain_path, "w", encoding="cp1252", errors="replace") as fout:
    for line in fin:
        fout.write(line)

print(f"Wrote {gaf_plain_path}")

Wrote goa_human.gaf


In [10]:
# Read the annotations and keep the Biological Process (BP) namespace
gafreader = GafReader(gaf_plain_path)
ns2assoc = gafreader.get_ns2assc()
bp = ns2assoc["BP"]
sample_key = next(iter(bp))
print(f"BP-annotated genes: {len(bp)}")
print(f"sample key: {sample_key!r}")

HMS:0:00:19.515936 906,445 annotations READ: goa_human.gaf 
BP-annotated genes: 31530
sample key: 'A0A075B6T8'


In [11]:
# Inspect one raw annotation row to find the field holding the gene symbol (DB_Symbol)
raw_assocs = gafreader.associations
print(f"total raw association rows: {len(raw_assocs)}")

first = raw_assocs[0]
print(type(first))
print(first._fields)   # namedtuple field names — look for something like DB_Symbol / DB_Object_Symbol
print(first)

total raw association rows: 906445
<class 'goatools.anno.init.reader_gaf.ntgafobj'>
('DB', 'DB_ID', 'DB_Symbol', 'Qualifier', 'GO_ID', 'DB_Reference', 'Evidence_Code', 'With_From', 'NS', 'DB_Name', 'DB_Synonym', 'DB_Type', 'Taxon', 'Date', 'Assigned_By', 'Extension', 'Gene_Product_Form_ID')
ntgafobj(DB='UniProtKB', DB_ID='A0A024RBG1', DB_Symbol='NUDT4B', Qualifier={'enables'}, GO_ID='GO:0005515', DB_Reference={'PMID:33961781'}, Evidence_Code='IPI', With_From={'UniProtKB:Q8NFP7'}, NS='MF', DB_Name={'Diphosphoinositol polyphosphate phosphohydrolase NUDT4B'}, DB_Synonym={'NUDT4B'}, DB_Type='protein', Taxon=[9606], Date=datetime.date(2026, 4, 25), Assigned_By='IntAct', Extension=None, Gene_Product_Form_ID={'UniProtKB:A0A024RBG1'})


In [12]:
# Gene symbol -> set of BP GO terms, skipping negative ('NOT') annotations
symbol2bp_go = defaultdict(set)

for row in raw_assocs:
    if row.NS != "BP":
        continue
    if any("NOT" in q for q in row.Qualifier):
        continue
    symbol2bp_go[row.DB_Symbol].add(row.GO_ID)

symbol2bp_go = dict(symbol2bp_go)
print(f"genes with >=1 BP annotation (by symbol): {len(symbol2bp_go)}")

genes with >=1 BP annotation (by symbol): 31557


In [13]:
# Regulon genes: Ensembl -> symbol, and how many have at least one BP annotation
# (exploratory; the next cell repeats the steps that are actually needed)
with open(os.path.join(output_dir, "regulons.json")) as f:
    regulons = json.load(f)

regulon_genes = set()
for genes in regulons.values():
    regulon_genes.update(genes)

print(f"total unique genes across all {CANCER} regulons: {len(regulon_genes)}")

idmap = pd.read_csv(idmap_path, sep="\t", header=0)  # same idmap_path your notebook 02 already defines
gene_name_map = idmap[idmap['Source'] == 'Gene Name'][['Preferred_Name', 'Name']]
ensembl_to_symbol = dict(zip(gene_name_map['Preferred_Name'], gene_name_map['Name']))

idmap = pd.read_csv(idmap_path, sep="\t", header=0)
gene_name_map = idmap[idmap['Source'] == 'Gene Name'][['Preferred_Name', 'Name']]
ensembl_to_symbol = dict(zip(gene_name_map['Preferred_Name'], gene_name_map['Name']))

regulons_symbol = {
    tf: [ensembl_to_symbol.get(g, g) for g in genes]
    for tf, genes in regulons.items()
}

regulon_genes_symbol = set()
for genes in regulons_symbol.values():
    regulon_genes_symbol.update(genes)

print(f"unique regulon genes (symbol): {len(regulon_genes_symbol)}")
covered = regulon_genes_symbol & symbol2bp_go.keys()
print(f"with >=1 BP annotation: {len(covered)} / {len(regulon_genes_symbol)} ({100*len(covered)/len(regulon_genes_symbol):.1f}%)")
print(f"unique regulon genes (symbol): {len(regulon_genes_symbol)}")
covered = regulon_genes_symbol & symbol2bp_go.keys()
print(f"with >=1 BP annotation: {len(covered)} / {len(regulon_genes_symbol)} ({100*len(covered)/len(regulon_genes_symbol):.1f}%)")

covered = regulon_genes & symbol2bp_go.keys()
missing = regulon_genes - symbol2bp_go.keys()

pct = 100 * len(covered) / len(regulon_genes)
print(f"regulon genes with >=1 BP annotation: {len(covered)} / {len(regulon_genes)} ({pct:.1f}%)")
print(f"sample of missing genes: {sorted(missing)[:20]}")

total unique genes across all GBM regulons: 6701
unique regulon genes (symbol): 6701
with >=1 BP annotation: 6372 / 6701 (95.1%)
unique regulon genes (symbol): 6701
with >=1 BP annotation: 6372 / 6701 (95.1%)
regulon genes with >=1 BP annotation: 0 / 6701 (0.0%)
sample of missing genes: ['ENSG00000000419', 'ENSG00000000457', 'ENSG00000000938', 'ENSG00000000971', 'ENSG00000001167', 'ENSG00000001497', 'ENSG00000001561', 'ENSG00000001617', 'ENSG00000001630', 'ENSG00000002016', 'ENSG00000002330', 'ENSG00000002549', 'ENSG00000002587', 'ENSG00000002746', 'ENSG00000002834', 'ENSG00000002919', 'ENSG00000003056', 'ENSG00000003137', 'ENSG00000003249', 'ENSG00000003400']


In [14]:
# Everything the GO enrichment needs, in one cell: regulons as symbols, symbol -> BP terms, background genes, GO DAG
idmap = pd.read_csv(idmap_path, sep="\t", header=0)
gene_name_map = idmap[idmap['Source'] == 'Gene Name'][['Preferred_Name', 'Name']]
ensembl_to_symbol = dict(zip(gene_name_map['Preferred_Name'], gene_name_map['Name']))

with open(os.path.join(output_dir, "regulons.json")) as f:
    regulons = json.load(f)

regulons_symbol = {tf: [ensembl_to_symbol.get(g, g) for g in genes] for tf, genes in regulons.items()}
regulon_genes_symbol = set(g for genes in regulons_symbol.values() for g in genes)
print(f"unique regulon genes (symbol): {len(regulon_genes_symbol)}")

gaf_plain_path = os.path.join(GO_DIR, "goa_human.gaf")

gafreader = GafReader(gaf_plain_path)
raw_assocs = gafreader.associations

symbol2bp_go = defaultdict(set)
for row in raw_assocs:
    if row.NS != "BP":
        continue
    if any("NOT" in q for q in row.Qualifier):
        continue
    symbol2bp_go[row.DB_Symbol].add(row.GO_ID)
symbol2bp_go = dict(symbol2bp_go)

print(f"genes with >=1 BP annotation (by symbol): {len(symbol2bp_go)}")

# Background = every gene expressed in this cancer (not just regulon genes)
background_genes = set(pd.read_csv(os.path.join(data_dir, f"{prefix}_zscored.csv"), index_col=0, usecols=[0]).index)
print(f"full {CANCER} expressed gene universe: {len(background_genes)}")

godag = GODag(os.path.join(GO_DIR, "go-basic.obo"))
print(f"GO DAG loaded: {len(godag)} terms")

unique regulon genes (symbol): 6701
HMS:0:00:19.012081 906,445 annotations READ: goa_human.gaf 
genes with >=1 BP annotation (by symbol): 31557
full GBM expressed gene universe: 9790
go-basic.obo: fmt(1.2) rel(2026-03-25) 41,853 Terms
GO DAG loaded: 41853 terms


In [15]:
# ORA (overrepresentation analysis) of GO enrichment terms. Answers the question "out of all GO terms this regulon's genes are
# annotated with, which occur signficantly more than by chance?"

# Calculates 4 values: how how many regulon genes have GO term X, how many regulon genes are there in total, how many background genes have term X 
# and how many background genes are there in total

# These 4 terms give raw p-value for each GO term. After correction for FDR, only significant terms remain

# Builds the background side of the ORA analysis so that it doesn't have to be calculated repeatedly:
goe_obj = GOEnrichmentStudy(

    # checks if this gene set is enriched with GO terms
    background_genes,

    # dict of the format {str: list} where key is HUGO symbol and value is [GO term 1, GO term 2, ...]
    symbol2bp_go,

    # DAG (directed acyclic graph) of GO terms w/ child-parent rels. E.g.: positive regulation of process X is a child term of process X
    godag,

    # Propagates up the DAG (basically if a gene is annoted with a child term, it is marked as annoted with parent terms all the way up the DAG)
    propagate_counts=True,

    # Significance threshold; p_val must be <= alpha
    alpha=0.05,

    # Benjamini-Hochberg correction for FDR
    methods=['fdr_bh']
)


Load  Ontology Enrichment Analysis ...
Propagating term counts up: is_a
 95%  9,291 of  9,790 population items found in association


In [16]:
# Helper fn: tracks GO terms in a particular geneset
def study_terms(study_genes, assoc):

    '''

    Collects every GO term that at least one gene of a gene set is annotated with.

    Args: study_genes (list). Gene symbols of one regulon
          assoc (dict). {gene symbol: set of GO term IDs}

    Returns: terms (set) GO term IDs annotated to any gene in study_genes

    '''

    terms = set()

    # Union of GO terms in each set
    for g in study_genes:
        terms |= assoc.get(g, set())
    return terms

# Helper fn: FDR correction with m (no.of genes in background) = total no. of terms across all genes not just regulon genes
def bh_adjust_with_true_m(pvals, m_true):

    '''

    Benjamini-Hochberg FDR correction where the number of tests m is given explicitly (all GO terms in the background)
    instead of the number of p-values passed in (only the terms the regulon's genes carry).

    Args: pvals (list), length n_terms_tested. Raw p-values
          m_true (int). Number of GO terms in the whole background population

    Returns: out (np.array), shape (n_terms_tested,). q-values in the same order as pvals

    '''

    # Sort p-values in ascending order
    order = np.argsort(pvals)
    ranked = np.array(pvals)[order]

    # BH: q = p * m / rank
    q = ranked * m_true / (np.arange(len(ranked)) + 1)

    # Enforce monotonicity: a larger p-value can't get a smaller q-value
    # Then cap at 1
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)

    # Put q-values back in the original order
    out = np.empty_like(q)
    out[order] = q
    return out

# Setup
checkpoint_every = 500
checkpoint_path = os.path.join(output_dir, "go_bio_symbol_checkpoint.pkl")
final_path      = os.path.join(output_dir, "go_bio_symbol.pkl")

# Takes ~45 min, so reuse the saved result if it exists (delete go_bio_symbol.pkl to recompute)
if os.path.exists(final_path):
    with open(final_path, "rb") as f:
        go_bio_symbol = pickle.load(f)
    print(f"Loaded cached go_bio_symbol.pkl: {len(go_bio_symbol)} regulons with significant BP terms")

else:
    go_bio_symbol = {}
    m_true = len(goe_obj.go2popitems)
    print(f"m_true (population term count): {m_true}")
    print(f"regulons to process: {len(regulons_symbol)}")

    start = time.time()

    # Main loop:
    for i, (regulon_id, genes) in enumerate(regulons_symbol.items()):
        # Regulon genes present in the background; regulons with < 3 are skipped
        study_genes = [g for g in genes if g in background_genes]
        if len(study_genes) < 3:
            continue

        # Only test GO terms that at least one regulon gene is annotated with
        sel_terms = study_terms(study_genes, symbol2bp_go)
        if not sel_terms:
            continue

        results = goe_obj.run_study(study_genes, prt=None, selected_goids=sel_terms)
        if not results:
            continue

        # FDR-correct against the full background term count and keep terms with q <= 0.05
        raw_p = [r.p_uncorrected for r in results]
        qvals = bh_adjust_with_true_m(raw_p, m_true)
        sig_terms = [r.GO for r, q in zip(results, qvals) if q <= 0.05]
        if sig_terms:
            go_bio_symbol[regulon_id] = sig_terms

        if (i + 1) % checkpoint_every == 0:
            elapsed = time.time() - start
            print(f"{i+1}/{len(regulons_symbol)} done, {elapsed/60:.1f} min elapsed, {len(go_bio_symbol)} regulons with sig terms so far")
            with open(checkpoint_path, "wb") as f:
                pickle.dump(go_bio_symbol, f)

    # Save:
    print(f"DONE: {len(go_bio_symbol)} / {len(regulons_symbol)} regulons with significant BP terms, total time {(time.time()-start)/60:.1f} min")
    with open(final_path, "wb") as f:
        pickle.dump(go_bio_symbol, f)

Loaded cached go_bio_symbol.pkl: 405 regulons with significant BP terms


In [17]:
# 10 hallmarks of cancer (Hanahan-Weinberg)
# Rough search; pulls out all terms in the GO DAG that contain certain keywords related to each hallmark

hallmarks_updated = {
    'SelfSufficiencyInGrowthSignals':      {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['prolifer', 'growth factor', 'mitogen', 'cell growth'])},
    'InsensitivityToAntigrowthSignals':    {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['cell cycle', 'growth inhibit', 'antiproliferat', 'cyclin', 'tumor suppress'])},
    'EvadingApoptosis':                    {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['apoptot', 'apoptosis', 'programmed cell death', 'survival'])},
    'LimitlessReplicativePotential':       {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['telomer', 'replicat', 'immortal', 'cell division'])},
    'SustainedAngiogenesis':               {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['angiogenes', 'vasculogenes', 'vessel', 'vascular'])},
    'TissueInvasionAndMetastasis':         {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['invasion', 'metastas', 'migration', 'motility', 'extracellular matrix'])},
    'GenomeInstabilityAndMutation':        {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['dna repair', 'genome instab', 'mutat', 'dna damage', 'chromosome instab'])},
    'TumorPromotingInflammation':          {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['inflammat', 'immune respon', 'cytokine', 'nf-kb', 'interleukin'])},
    'ReprogrammingEnergyMetabolism':       {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['glycolys', 'oxidative phosphorylat', 'metabol', 'atp', 'glucose'])},
    'EvadingImmuneDetection':              {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['immune evasion', 'immune escape', 'immunosuppress', 't cell', 'mhc'])},
}

# Show the number of GO terms associated w/ each of the 10 hallmarks
for h, terms in hallmarks_updated.items():
    print(f"{h}: {len(terms)} GO terms")

# Computes term rarity based on how many regulons are enriched for each term
term_counts = TermCounts(godag, go_bio_symbol)  

# Every unique GO term that the regulons are enriched for
regulon_go_terms  = list({t for terms in go_bio_symbol.values() for t in terms if t in godag})

# Every unique GO term associated with the cancer hallmarks
hallmark_go_terms = list({t for terms in hallmarks_updated.values() for t in terms if t in godag})

print(f"Unique regulon GO terms:  {len(regulon_go_terms)}")
print(f"Unique hallmark GO terms: {len(hallmark_go_terms)}")

SelfSufficiencyInGrowthSignals: 619 GO terms
InsensitivityToAntigrowthSignals: 205 GO terms
EvadingApoptosis: 380 GO terms
LimitlessReplicativePotential: 330 GO terms
SustainedAngiogenesis: 197 GO terms
TissueInvasionAndMetastasis: 424 GO terms
GenomeInstabilityAndMutation: 75 GO terms
TumorPromotingInflammation: 852 GO terms
ReprogrammingEnergyMetabolism: 1207 GO terms
EvadingImmuneDetection: 707 GO terms
Unique regulon GO terms:  1176
Unique hallmark GO terms: 4714


In [18]:
# Now doing Lin semantic similarity: for every term in regulon_go_terms, it asks how similar it is to the terms in hallmark_go_terms
# Returns a score b/w 0 (fully unrelated) and 1 (identical)

# Prints output every x interactions
checkpoint_every = 250

checkpoint_path = os.path.join(output_dir, "lin_cache_checkpoint.pkl")
final_path      = os.path.join(output_dir, "lin_cache.pkl")

# Takes ~15 min for BRCA, so reuse the saved result if it exists (delete lin_cache.pkl to recompute)
if os.path.exists(final_path):
    with open(final_path, "rb") as f:
        lin_cache = pickle.load(f)
    print(f"Loaded cached lin_cache.pkl: {len(lin_cache)} regulon terms")

else:
    # Lookup table
    lin_cache = {}
    start = time.time()

    # Goes through each of the regulon terms
    for i, t1 in enumerate(regulon_go_terms):

        sims = {}

        # For each regulon term, it goes through each of the hallmark terms
        for t2 in hallmark_go_terms:

            try:
                # Gets similarity score between the current t1 (regulon term) and current t2 (hallmark term)
                # term_counts contains term rarities based on how many regulons are enriched
                s = lin_sim(t1, t2, godag, term_counts)

                # Safety net; if similarity returns None, defaults to 0
                sims[t2] = s if s else 0

            # Safety net but if similarity fails
            except:
                sims[t2] = 0

        # sims is a dict of the form {hallmark term: similarity score} for the current regulon term (t1)
        lin_cache[t1] = sims

        # Periodically displays progress and saves for later
        if (i + 1) % checkpoint_every == 0:
            elapsed = time.time() - start
            print(f"{i+1}/{len(regulon_go_terms)} done, {elapsed/60:.1f} min elapsed")
            with open(checkpoint_path, "wb") as f:
                pickle.dump(lin_cache, f)

    # Finally, lin_cache is a nested dict of the format {regulon term : {hallmark term 1: sim score 1, hallmark term 2: sim score 2, ...}}
    print(f"DONE: {len(lin_cache)} regulon terms cached, total time {(time.time()-start)/60:.1f} min")
    with open(final_path, "wb") as f:
        pickle.dump(lin_cache, f)

Loaded cached lin_cache.pkl: 1176 regulon terms


In [19]:
# Takes one regulon id, its list of sig GO terms and a similarity threshold (here it is 80%)
# regulon_go_terms[regulon_id] = enriched_go
def regulon_hallmark_sim_fast(reg_id, enriched_go, threshold=0.8):

    '''

    Finds which cancer hallmarks a regulon is enriched for. A hallmark is passed if any of the regulon's GO terms has
    Lin similarity >= threshold with any GO term of that hallmark. Uses lin_cache, hallmarks_updated and godag from above.

    Args: reg_id (str). Regulon ID
          enriched_go (list). Significant GO term IDs of the regulon
          threshold (float). Minimum Lin similarity to count as a match (default 0.8)

    Returns: reg_id (str), passed (list) names of the hallmarks the regulon qualifies for

    '''

    # collects hallmark CATEGORIES that the regulon qualifies for
    passed = []

    # Drops any GO terms not present in lin_cache, which contains all GO terms for which regulon is enriched
    enriched_go = [t for t in enriched_go if t in lin_cache]

    # Loops through each of the 10 Hanahan-Weinberg hallmarks
    # hallmarks_updated is dict of the form {hallmark: [go_terms]}
    for hallmark, go_terms in hallmarks_updated.items():

        # Safety check: only takes hallmark terms present in the overall GO DAG
        hallmark_terms = [t for t in go_terms if t in godag]

        
        max_sim = 0

        # Highest similarity between any regulon term and any term of this hallmark (stops early once it hits 1.0)
        for t1 in enriched_go:
            for t2 in hallmark_terms:
                s = lin_cache[t1].get(t2, 0)
                if s > max_sim:
                    max_sim = s
                if max_sim == 1.0:
                    break
            if max_sim == 1.0:
                break
        if max_sim >= threshold:
            passed.append(hallmark)
    return reg_id, passed

# Apply to every regulon with significant GO terms: {regulon ID: [hallmarks passed]}
hallmark_enriched = {}
for reg_id, go_terms in go_bio_symbol.items():
    if len(go_terms) == 0:
        continue
    _, passed = regulon_hallmark_sim_fast(reg_id, list(go_terms))
    if passed:
        hallmark_enriched[reg_id] = passed

print("Regulons with hallmark enrichment:", len(hallmark_enriched))

all_hallmarks = [h for hallmarks in hallmark_enriched.values() for h in hallmarks]
print(Counter(all_hallmarks).most_common())

Regulons with hallmark enrichment: 343
[('TumorPromotingInflammation', 182), ('ReprogrammingEnergyMetabolism', 168), ('TissueInvasionAndMetastasis', 108), ('EvadingImmuneDetection', 104), ('LimitlessReplicativePotential', 58), ('SustainedAngiogenesis', 53), ('SelfSufficiencyInGrowthSignals', 45), ('EvadingApoptosis', 39), ('InsensitivityToAntigrowthSignals', 33), ('GenomeInstabilityAndMutation', 31)]


In [20]:
# Combine the two criteria: survival-significant OR hallmark-enriched
survival_series = pd.read_csv(os.path.join(output_dir, "disease_relevant_regulons_survival_only.csv"), header=None)[0]
survival_set = set(survival_series.astype(str))
hallmark_set = set(str(k) for k in hallmark_enriched.keys())

overlap = survival_set & hallmark_set
survival_or_hallmark = survival_set | hallmark_set

print(f"Survival significant: {len(survival_set)}")
print(f"Hallmark enriched: {len(hallmark_set)}")
print(f"Overlap (both criteria): {len(overlap)}")
print(f"Survival OR hallmark (final disease-relevant set): {len(survival_or_hallmark)}")

pd.Series(sorted(survival_or_hallmark, key=int)).to_csv(
    os.path.join(output_dir, "final_disease_relevant_regulons.csv"), index=False, header=False)

print(f"Saved {len(survival_or_hallmark)} disease-relevant regulons")

Survival significant: 1354
Hallmark enriched: 343
Overlap (both criteria): 161
Survival OR hallmark (final disease-relevant set): 1536
Saved 1536 disease-relevant regulons


In [21]:
# Keep the intermediate (survival OR hallmark) set in its own file.
# final_disease_relevant_regulons.csv gets overwritten further down, once the VAL_NAME co-expression validation is done
# (GBM equivalent: final = (survival OR hallmark) AND Gravendeel co-expression validated)
pd.Series(sorted(survival_or_hallmark, key=int)).to_csv(
    os.path.join(output_dir, "disease_relevant_regulons_survival_or_hallmark.csv"),
    index=False, header=False
)
print(f"Saved {len(survival_or_hallmark)} survival-OR-hallmark regulons")

Saved 1536 survival-OR-hallmark regulons


## Co-expression validation in an independent cohort (VAL_NAME)

GBM used Gravendeel for this step; for each cancer the independent cohort is **VAL_NAME** (see the config cell; skipped if the cancer has none).

A regulon counts as validated if its genes are still co-expressed in VAL_NAME:
1. PC1 of the regulon's genes explains more variance than 95% of random gene sets of the same size (permutation p ≤ 0.05), **and**
2. PC1 variance explained ≥ 0.3 (hard cutoff from the paper)

Final disease-relevant regulons = (survival OR hallmark) AND VAL_NAME-validated.

In [22]:
# Load the validation cohort: expression with one row per gene symbol, plus clinical data if available
# Formats handled: cBioPortal studies (e.g. METABRIC), GSE72094 (GEO SOFT file), E-MTAB-1980 (ArrayExpress) and Gravendeel (GSE16011)
# For any other cohort, add a branch that builds ext_raw (gene symbols x samples, log2 scale) and, if available, ext_clin
if HAS_VALIDATION:

    ext_dir = val_dir
    ext_clin = None


    if VAL_NAME == "GSE72094":

        # GEO "SOFT family" file: one platform block (probe ID -> gene symbol) followed by one block per sample,
        # each with clinical fields (!Sample_characteristics_ch1 = name: value) and a table of probe ID / log2 expression
        # Parsing the 250 MB file takes a few minutes, so the two tidy tables are cached next to it (delete them to rebuild)
        soft_file = os.path.join(ext_dir, "GSE72094_family.soft.gz")
        expr_csv  = os.path.join(ext_dir, "gse72094_expression_by_symbol.csv.gz")
        clin_csv  = os.path.join(ext_dir, "gse72094_clinical_raw.csv")

        if not (os.path.exists(expr_csv) and os.path.exists(clin_csv)):
            assert os.path.exists(soft_file), f"missing {soft_file}"
            probe_ids, probe_to_symbol, probe_pos, samples, clin_rows = [], {}, None, {}, {}
            block, in_table, sample_id = None, False, None

            with gzip.open(soft_file, "rt", errors="replace") as f:
                for line in f:
                    line = line.rstrip("\n")

                    # A line starting with ^ opens a new block (platform, or one sample)
                    if line.startswith("^"):
                        block, in_table = line.split(" = ")[0], False
                        if block == "^SAMPLE":
                            # The platform block comes first, so the full probe list is known by now
                            # One float32 array per sample (position = probe) keeps memory low: ~60k probes x 442 samples
                            if probe_pos is None:
                                probe_pos = {p: i for i, p in enumerate(probe_ids)}
                            sample_id = line.split(" = ")[1]
                            samples[sample_id] = np.full(len(probe_ids), np.nan, dtype=np.float32)
                            clin_rows[sample_id] = {}
                        continue

                    # Table boundaries inside a block
                    if line.startswith("!platform_table_begin") or line.startswith("!sample_table_begin"):
                        in_table, header_seen = True, False
                        continue
                    if line.startswith("!platform_table_end") or line.startswith("!sample_table_end"):
                        in_table = False
                        continue

                    if in_table:
                        # First row of every table is its header
                        if not header_seen:
                            header_seen = True
                            continue
                        parts = line.split("\t")

                        # Platform table columns: ID, GB_LIST, EntrezGeneID, GeneSymbol, SPOT_ID
                        if block == "^PLATFORM":
                            probe_ids.append(parts[0])
                            if len(parts) >= 4 and parts[3].strip():
                                probe_to_symbol[parts[0]] = parts[3].strip()

                        # Sample table columns: ID_REF, VALUE
                        elif block == "^SAMPLE" and len(parts) >= 2 and parts[0] in probe_pos:
                            try:
                                samples[sample_id][probe_pos[parts[0]]] = float(parts[1])
                            except ValueError:
                                pass

                    # Clinical fields of the current sample
                    elif block == "^SAMPLE" and line.startswith("!Sample_characteristics_ch1 = ") and ": " in line:
                        key, value = line[len("!Sample_characteristics_ch1 = "):].split(": ", 1)
                        clin_rows[sample_id][key.strip()] = value.strip()

            # Probes x samples, keeping probes that map to exactly one gene symbol, then one row per symbol (mean of its probes)
            probes = pd.DataFrame(samples, index=probe_ids)
            del samples
            symbols = pd.Series(probe_to_symbol).reindex(probes.index)
            single = symbols.notna() & ~symbols.fillna("").str.contains(r"[ /,;|]")
            print(f"Probes mapped to a single gene symbol: {int(single.sum())} / {len(probes)}")
            by_symbol = probes[single].groupby(symbols[single]).mean()
            by_symbol.index.name = "Hugo_Symbol"
            by_symbol.to_csv(expr_csv)
            pd.DataFrame(clin_rows).T.rename_axis("PATIENT_ID").to_csv(clin_csv)

        ext_raw = pd.read_csv(expr_csv, index_col=0)

        # Overall survival: vital_status (Alive / Dead) and survival_time_in_days; patients with either missing are dropped
        clin_geo = pd.read_csv(clin_csv, index_col=0)
        ext_clin = pd.DataFrame({
            "OS_MONTHS": pd.to_numeric(clin_geo["survival_time_in_days"], errors="coerce") / 30.4375,
            "OS_STATUS": clin_geo["vital_status"].astype(str).str.strip().str.lower().map({"alive": 0, "dead": 1}),
        }).dropna()
        ext_clin = ext_clin[ext_clin["OS_MONTHS"] >= 0]
        ext_clin["OS_STATUS"] = ext_clin["OS_STATUS"].astype(int)

    elif VAL_NAME == "EMTAB1980":

        # Processed matrix from ArrayExpress: one row per microarray probe, log2 quantile-normalised
        # Columns: REF (Agilent probe ID), Feature Numbers, SystematicName (mostly RefSeq mRNA IDs, e.g. NM_004333), then one column per tumour
        expr_file = os.path.join(ext_dir, "ccRCC_exp_log_quantile_normalized.txt")
        assert os.path.exists(expr_file), f"missing {expr_file}"
        probes = pd.read_csv(expr_file, sep="\t")
        sample_cols = [c for c in probes.columns if c.startswith("ccRCC-")]

        # Probe -> gene symbol, using miner's mapping table: RefSeq mRNA ID -> Ensembl gene ID -> gene symbol
        idmap = pd.read_csv(idmap_path, sep="\t", header=0)
        refseq = idmap[idmap["Source"] == "RefSeq mRNA ID"]
        names  = idmap[idmap["Source"] == "Gene Name"]
        refseq_to_ensembl = dict(zip(refseq["Name"], refseq["Preferred_Name"]))
        ensembl_to_symbol = dict(zip(names["Preferred_Name"], names["Name"]))
        probes["Hugo_Symbol"] = probes["SystematicName"].map(refseq_to_ensembl).map(ensembl_to_symbol)
        print(f"Probes mapped to a gene symbol: {int(probes['Hugo_Symbol'].notna().sum())} / {len(probes)}")

        # Several probes can measure the same gene: average them into one row per symbol
        ext_raw = probes.dropna(subset=["Hugo_Symbol"]).groupby("Hugo_Symbol")[sample_cols].mean()

        # Survival is NOT in the ArrayExpress files (the sample sheet only has sex). If a table built from the paper's
        # supplementary data is saved as emtab1980_clinical.csv (columns PATIENT_ID, OS_MONTHS, OS_STATUS 0/1), it is used here
        clin_csv = os.path.join(ext_dir, "emtab1980_clinical.csv")
        if os.path.exists(clin_csv):
            ext_clin = pd.read_csv(clin_csv).dropna(subset=["PATIENT_ID", "OS_MONTHS", "OS_STATUS"])
            ext_clin = ext_clin[ext_clin["OS_MONTHS"] >= 0].set_index("PATIENT_ID")

    elif VAL_NAME == "GRAVENDEEL":

        # Gravendeel et al. 2009 (GSE16011): Affymetrix arrays of gliomas of all grades plus a few control brain samples
        # gravendeel_expression.csv: RMA-normalised log2 expression, one row per Entrez gene ID (made in R with the ENTREZG
        # custom CDF, as in the paper), one column per GEO sample (GSM...)
        expr_file = os.path.join(ext_dir, "gravendeel_expression.csv")
        soft_file = os.path.join(ext_dir, "GSE16011_family.soft.gz")
        clin_csv  = os.path.join(ext_dir, "gravendeel_clinical.csv")
        assert os.path.exists(expr_file), f"missing {expr_file}"
        assert os.path.exists(soft_file), f"missing {soft_file}"
        by_entrez = pd.read_csv(expr_file, index_col=0, low_memory=False)

        # Entrez gene ID -> Ensembl ID -> gene symbol, using miner's mapping table
        idmap = pd.read_csv(idmap_path, sep="\t", header=0)
        entrez = idmap[idmap["Source"] == "Entrez Gene ID"]
        names  = idmap[idmap["Source"] == "Gene Name"]
        entrez_to_ensembl = dict(zip(entrez["Name"].astype(str).str.replace(r"\.0$", "", regex=True), entrez["Preferred_Name"]))
        ensembl_to_symbol = dict(zip(names["Preferred_Name"], names["Name"]))
        row_ids = by_entrez.index.astype(str).str.replace("_at", "", regex=False)
        by_entrez["Hugo_Symbol"] = row_ids.map(entrez_to_ensembl).map(ensembl_to_symbol)
        print(f"Genes mapped to a symbol: {int(by_entrez['Hugo_Symbol'].notna().sum())} / {len(by_entrez)}")
        ext_raw = by_entrez.dropna(subset=["Hugo_Symbol"]).groupby("Hugo_Symbol").mean(numeric_only=True)

        # GEO sample ID -> the paper's sample number, from the sample titles ("glioma 8", "control 7") in the SOFT file
        gsm_to_number, is_glioma, current = {}, {}, None
        with gzip.open(soft_file, "rt", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.strip()
                if line.startswith("^SAMPLE"):
                    current = line.split("=")[1].strip()
                elif line.startswith("!Sample_title") and current:
                    title = line.split("=")[1].strip()
                    digits = "".join(ch for ch in title if ch.isdigit())
                    if digits:
                        gsm_to_number[current] = digits
                        is_glioma[current] = title.lower().startswith("glioma")

        # Keep the glioma samples and name them by sample number, which is how the survival table is indexed
        ext_raw.columns = [col.replace(".CEL", "") for col in ext_raw.columns]
        ext_raw = ext_raw.loc[:, [col for col in ext_raw.columns if is_glioma.get(col, False)]]
        ext_raw.columns = [gsm_to_number[col] for col in ext_raw.columns]
        ext_raw = ext_raw.loc[:, ~ext_raw.columns.duplicated()]
        print(f"Glioma samples: {ext_raw.shape[1]}")

        # Survival typed in from the paper's supplementary table (columns sample_id, OS_MONTHS, OS_STATUS 0/1)
        if os.path.exists(clin_csv):
            ext_clin = pd.read_csv(clin_csv, dtype={"sample_id": str}).dropna(subset=["sample_id", "OS_MONTHS", "OS_STATUS"])
            ext_clin = ext_clin[ext_clin["OS_MONTHS"] >= 0].set_index("sample_id")
            ext_clin.index.name = "PATIENT_ID"
            ext_clin["OS_STATUS"] = ext_clin["OS_STATUS"].astype(int)

    else:

        # cBioPortal format: extract all relevant mRNA expression files
        expr_files = [f for f in glob.glob(os.path.join(ext_dir, "**", "data_mrna*.txt"), recursive=True)
                      if "zscore" not in os.path.basename(f).lower()]

        print("Expression file:", expr_files)
        assert expr_files, f"no data_mrna*.txt found in {ext_dir}"

        ext_raw = pd.read_csv(expr_files[0], sep="\t")

        # Drop extra gene symbol column (Entrez ID)
        ext_raw = ext_raw.drop(columns=[c for c in ["Entrez_Gene_Id"] if c in ext_raw.columns])

        # Collapse duplicate symbols
        ext_raw = ext_raw.dropna(subset=["Hugo_Symbol"]).groupby("Hugo_Symbol").mean(numeric_only=True)

        # Load clinical data; header lines starting with '#' are dropped
        clin_file = glob.glob(os.path.join(ext_dir, "**", "data_clinical_patient.txt"), recursive=True)[0]
        ext_clin_all = pd.read_csv(clin_file, sep="\t", comment="#")

        # Some cBioPortal studies (e.g. CPTAC) record no survival; then only the co-expression validation can use the cohort
        if not {"OS_MONTHS", "OS_STATUS"}.issubset(ext_clin_all.columns):
            ext_clin = None
        else:
            # Keep all columns for now (needed for disease-specific survival below), then extract the OS columns
            ext_clin = ext_clin_all[["PATIENT_ID", "OS_MONTHS", "OS_STATUS"]].copy()

            # Text to binary, dropping all NaN and negative survival time values
            ext_clin["OS_STATUS"] = pd.to_numeric(ext_clin["OS_STATUS"].astype(str).str.split(":").str[0], errors="coerce")
            ext_clin["OS_MONTHS"] = pd.to_numeric(ext_clin["OS_MONTHS"], errors="coerce")
            ext_clin = ext_clin.dropna()
            ext_clin = ext_clin[ext_clin["OS_MONTHS"] >= 0].set_index("PATIENT_ID")

            # Disease-specific survival when the cohort records cause of death (e.g. METABRIC: VITAL_STATUS)
            # Missing cause of death -> NaN (not 0), so a death with unknown cause isn't counted as censored
            if "VITAL_STATUS" in ext_clin_all.columns:
                vs = ext_clin_all.set_index("PATIENT_ID")["VITAL_STATUS"].reindex(ext_clin.index)
                ext_clin["DSS_STATUS"] = np.where(vs.isna(), np.nan, (vs == "Died of Disease").astype(float))
                print(f"Disease-specific deaths: {int(ext_clin['DSS_STATUS'].sum())} | unknown cause: {int(ext_clin['DSS_STATUS'].isna().sum())}")

    print(f"{VAL_NAME} raw:", ext_raw.shape)

    # Drop genes missing in >10% of samples, impute remaining NaNs with the mean gene exprn value
    ext_raw = ext_raw.loc[ext_raw.isna().mean(axis=1) <= 0.10]
    ext_raw = ext_raw.T.fillna(ext_raw.mean(axis=1)).T

    # Z-score per gene
    ext_std = ext_raw.std(axis=1)
    ext_raw = ext_raw.loc[ext_std > 0]
    ext_z   = ((ext_raw.T - ext_raw.mean(axis=1)) / ext_std[ext_std > 0]).T
    print(f"{VAL_NAME} z-scored:", ext_z.shape, "| NaN:", int(ext_z.isna().sum().sum()))

    # Saved for notebook 05 (pickle is much faster than a large csv)
    ext_z.to_pickle(os.path.join(output_dir, f"{val}_zscored.pkl"))

    if ext_clin is not None:
        # Diagnostic: should be close to 100% overlap
        print(f"{VAL_NAME} clinical: {ext_clin.shape} | OS events: {int(ext_clin['OS_STATUS'].sum())} | "
              f"overlap with expression: {len(set(ext_clin.index) & set(ext_z.columns))}")
        ext_clin.to_csv(os.path.join(output_dir, f"{val}_clinical.csv"))
    else:
        print(f"{VAL_NAME}: no survival data found. Co-expression validation (this notebook) works without it; "
              f"notebook 05's external survival test needs {val}_clinical.csv")
else:
    print("No independent cohort for this cancer: co-expression validation skipped")


Genes mapped to a symbol: 16435 / 17589
Glioma samples: 276
GRAVENDEEL raw: (16427, 276)
GRAVENDEEL z-scored: (16427, 276) | NaN: 0
GRAVENDEEL clinical: (272, 2) | OS events: 239 | overlap with expression: 272


In [23]:
# Map regulons (member genes have Ensembl IDs) to symbols and restrict to genes measured in VAL_NAME
if HAS_VALIDATION:
    idmap = pd.read_csv(idmap_path, sep="\t", header=0)
    gene_name_map = idmap[idmap['Source'] == 'Gene Name'][['Preferred_Name', 'Name']]
    ensembl_to_symbol = dict(zip(gene_name_map['Preferred_Name'], gene_name_map['Name']))

    with open(os.path.join(output_dir, "regulons.json")) as f:
        regulons = json.load(f)

    ext_genes = set(ext_z.index)
    regulon_modules_ext = {}
    for reg_id, genes in regulons.items():
        syms = list(dict.fromkeys(ensembl_to_symbol.get(g, g) for g in genes))

        # Stores regulon ids that are present in validation cohort
        regulon_modules_ext[int(reg_id)] = [s for s in syms if s in ext_genes]

    all_reg_genes = set(ensembl_to_symbol.get(g, g) for genes in regulons.values() for g in genes)
    print(f"Regulon genes measured in {VAL_NAME}: {len(all_reg_genes & ext_genes)} / {len(all_reg_genes)}")

    # PCA needs a few genes; regulons with <3 genes in VAL_NAME can't be tested and count as NOT validated
    # Testable is dict of the format regulon_id: regulon_genes for all regulons in validation cohort with >= 3 genes
    testable = {r: g for r, g in regulon_modules_ext.items() if len(g) >= 3}
    print(f"Testable regulons (>=3 genes in {VAL_NAME}): {len(testable)} / {len(regulons)}")

    # program_coherence_ext has columns:

    # Modules (regulon_id)
    # N (how many regulon genes in validation cohort),
    # Variance (variance of all regulon genes within 1 patient, averaged over patients)
    # Variance_expained (Share of variance explained by PC1)
    program_coherence_ext = regulon_variance(ext_z, testable)

    # regulon ids, in row order
    coh_ids = program_coherence_ext.index.to_numpy()

    # Drop index
    program_coherence_ext = program_coherence_ext.reset_index(drop=True)      
    print(program_coherence_ext.head())
else:
    print("No independent cohort for this cancer: co-expression validation skipped")


Regulon genes measured in GRAVENDEEL: 6553 / 6701
Testable regulons (>=3 genes in GRAVENDEEL): 3961 / 3961
      N  Variance  Variance_expained  Modules
0  31.0  0.789521           0.239332      0.0
1  15.0  0.288976           0.715854      1.0
2   8.0  0.720675           0.367718      2.0
3   9.0  0.778153           0.246372      3.0
4   9.0  0.330505           0.682497      4.0


In [24]:
# Null distribution: 
if HAS_VALIDATION:

    # For each regulon size n, 250 groups of n genes from validation cohort are taken randomly and PC1 is computed
    # If a given regulon's PC1 is higher than 95% of these gene sets (1- 0.95 = 0.05= p) and PC1 >= 0.3, it is coexpression-validated

    # Differences from GBM (which made one serial call with n_iter=500):

    # 250 draws per size
    # 1 miner call per regulon size, run in parallel across CPU cores
    # Reproducible: each size gets a fixed seed (SEED_BASE + size) before calling miner with seed=False,
    # Results are identical on every rerun regardless of which core runs which size
    # Cached to csv so it only runs once (delete the csv to recompute).

    N_ITER    = 250
    SEED_BASE = 12000

    def _null_for_size(A, genes, n, n_iter, seed_base):

        '''

        Null distribution for one regulon size: draws n_iter random gene sets of n genes from the validation cohort and
        computes variance and PC1 variance explained for each. Runs in a separate worker process, hence the imports inside.

        Args: A (np.array), shape (n_genes, n_samples). Z-scored validation expression
              genes (list), length n_genes. Gene symbols, row labels of A
              n (int). Regulon size (number of genes per random set)
              n_iter (int). Number of random gene sets to draw
              seed_base (int). Random seed is seed_base + n, so each size is reproducible

        Returns: df, shape (n_iter, 3). Columns N, Variance, Variance_explained; one row per random gene set

        '''

        import numpy as np
        import pandas as pd
        from miner.miner import random_regulon_variance

        # A: entire expression matrix flattened into one array
        df = pd.DataFrame(A, index=genes)

        # Different random seed for each size, making it reproducible AND independent
        np.random.seed(seed_base + n)

        # Returns this df for each size : n 
        # Columns are variance, PC1 variance-explained
        # Shape: (1, 250) i.e (len([n], n_iter)
        return random_regulon_variance(df, [n], n_iter=n_iter, seed=False)

    ns_list   = sorted(program_coherence_ext["N"].astype(int).unique())
    rand_path = os.path.join(output_dir, f"{val}_random_regulon_variance.csv")
    print(f"Distinct regulon sizes to simulate: {len(ns_list)} | draws per size: {N_ITER} | CPU cores: {os.cpu_count()}")

    # If already run, load the cached null distribution
    if os.path.exists(rand_path):
        random_results_ext = pd.read_csv(rand_path, index_col=0)
        print("Loaded cached null distribution:", random_results_ext.shape)

    # Fresh run:
    else:
        start = time.time()

        # Expression values as a plain float array (cheap to send to worker processes) and the gene names
        A     = np.ascontiguousarray(ext_z.values, dtype=np.float64)
        genes = list(ext_z.index)
        tasks = (delayed(_null_for_size)(A, genes, int(n), N_ITER, SEED_BASE)

                 # largest (slowest) sizes first for load balancing
                 for n in sorted(ns_list, reverse=True))   
    
        # Progress bar ticks once per regulon size; it starts slow (big sizes) and speeds up
        parts = list(tqdm(Parallel(n_jobs= N_JOBS, return_as="generator_unordered")(tasks),
                          total=len(ns_list), desc="regulon sizes"))

        # fixed order by size gives identical csv on rerun
        parts.sort(key=lambda d: int(d.index[0]))              
        random_results_ext = pd.concat(parts)
        random_results_ext.to_csv(rand_path)
        print(f"Null distribution done in {(time.time()-start)/60:.1f} min:", random_results_ext.shape)

    # BOTH VARIANCE AND PC1 must be above 95% of null distbn. members. Below 2 lines calculate those thresholds
    # Below 2 dfs are of the format row --> regulon size, column --> cutoff value of PC1/ variance
    variance_explained_cutoffs = random_significance(random_results_ext, variable="Variance_explained", p=0.05)
    variance_cutoffs = random_significance(random_results_ext, variable="Variance", p=0.05)

    # Marks each regulon as significant or not (1/0) based on PC1, variance as compared to NULL DISTBN. not paper threshold
    var_pass, var_exp_pass, var_passDF, var_exp_passDF = significant_regulons(
        program_coherence_ext, variance_cutoffs, variance_explained_cutoffs)

    # Map row positions back to regulon ids; NOW APPLY PAPER THRESHOLD of 0.3
    # Convert 1/0 to True/ False
    perm_pass = np.array(var_exp_pass) == 1
    pc1_pass  = (program_coherence_ext["Variance_expained"] >= 0.3).to_numpy()
    validated_regulons = coh_ids[perm_pass & pc1_pass]

    print("Regulons passing permutation p<=0.05:", int(perm_pass.sum()))
    print("Regulons passing PC1>=0.3 AND p<=0.05:", len(validated_regulons))

    # Outputs:
    coherence_out = program_coherence_ext.assign(Regulon=coh_ids, Perm_pass=perm_pass.astype(int), PC1_pass=pc1_pass.astype(int))
    coherence_out.to_csv(os.path.join(output_dir, f"{val}_regulon_coherence.csv"), index=False)

    pd.Series(sorted(validated_regulons)).to_csv(
        os.path.join(output_dir, f"{val}_coexp_validated_regulons.csv"), index=False, header=False)

    print(f"Saved {val}_coexp_validated_regulons.csv")
else:
    print("No independent cohort for this cancer: co-expression validation skipped")


Distinct regulon sizes to simulate: 65 | draws per size: 250 | CPU cores: 22


regulon sizes:   0%|          | 0/65 [00:00<?, ?it/s]

Null distribution done in 0.5 min: (16250, 3)
Regulons passing permutation p<=0.05: 2634
Regulons passing PC1>=0.3 AND p<=0.05: 2379
Saved gravendeel_coexp_validated_regulons.csv


In [25]:
# Final disease-relevant regulons = (survival OR hallmark) AND VAL_NAME co-expression validated  (same rule as GBM)
# Without an independent cohort the final set is (survival OR hallmark) only, and is marked as not externally validated
survival_set = set(pd.read_csv(os.path.join(output_dir, "disease_relevant_regulons_survival_only.csv"), header=None)[0].astype(str))
union_set    = set(pd.read_csv(os.path.join(output_dir, "disease_relevant_regulons_survival_or_hallmark.csv"), header=None)[0].astype(str))

print("Survival significant:", len(survival_set))
print("Survival OR hallmark:", len(union_set))

if HAS_VALIDATION:
    coexp_set = set(pd.read_csv(os.path.join(output_dir, f"{val}_coexp_validated_regulons.csv"), header=None)[0].astype(str))
    final_regulons = union_set & coexp_set
    status = f"externally validated: co-expression in {VAL_NAME}; final = (survival OR hallmark) AND validated"
    print(f"Co-expression validated ({VAL_NAME}):", len(coexp_set))
    print("  of which survival-significant:", len(survival_set & coexp_set))
else:
    final_regulons = union_set
    status = "NOT externally validated: no independent cohort; final = (survival OR hallmark) only"

print("Final disease-relevant regulons:", len(final_regulons))
print("Status:", status)
print("(GBM for comparison: 1310 survival, 298 hallmark, 1509 union, 1926 validated -> 907 final)")

# Written WITHOUT a header; always read back with header=None
pd.Series(sorted(final_regulons, key=int)).to_csv(
    os.path.join(output_dir, "final_disease_relevant_regulons.csv"), index=False, header=False)
print(f"Saved {len(final_regulons)} regulons to final_disease_relevant_regulons.csv")

# Records whether this cancer's results were checked in an independent cohort
with open(os.path.join(output_dir, "validation_status.txt"), "w") as f:
    f.write(status + "\n")


Survival significant: 1354
Survival OR hallmark: 1536
Co-expression validated (GRAVENDEEL): 2379
  of which survival-significant: 1154
Final disease-relevant regulons: 1292
Status: externally validated: co-expression in GRAVENDEEL; final = (survival OR hallmark) AND validated
(GBM for comparison: 1310 survival, 298 hallmark, 1509 union, 1926 validated -> 907 final)
Saved 1292 regulons to final_disease_relevant_regulons.csv


In [27]:
# Final criteria for disease relevance: survival association must also replicate in the validation cohort

if HAS_VALIDATION and os.path.exists(os.path.join(output_dir, f"{val}_clinical.csv")):

    # Validation cohort expression (gene symbols x samples, z-scored) and survival table
    ext_expr = pd.read_pickle(os.path.join(output_dir, f"{val}_zscored.pkl"))
    ext_expr.columns = ext_expr.columns.astype(str)
    ext_surv = pd.read_csv(os.path.join(output_dir, f"{val}_clinical.csv"), index_col=0)
    ext_surv.index = ext_surv.index.astype(str)
    ext_ids = [p for p in ext_expr.columns if p in ext_surv.index]

    # Gravendeel: the paper uses grade II, III and IV gliomas only, so grade I samples are left out
    if VAL_NAME == "GRAVENDEEL":
        histology, title = {}, None
        with gzip.open(os.path.join(val_dir, "GSE16011_family.soft.gz"), "rt", errors="ignore") as f:
            for line in f:
                if line.startswith("!Sample_title"):
                    title = line.split("=", 1)[1].strip()
                elif line.startswith("!Sample_characteristics_ch1 = histology:"):
                    histology[title.split()[-1]] = line.split("histology:", 1)[1].strip()
        ext_ids = [p for p in ext_ids if histology.get(p, "").endswith(("(grade II)", "(grade III)", "(grade IV)"))]

    ext_surv = ext_surv.loc[ext_ids, ["OS_MONTHS", "OS_STATUS"]]
    print(f"{VAL_NAME} patients used: {len(ext_ids)} | deaths: {int(ext_surv['OS_STATUS'].sum())}")

    # Regulon genes as symbols
    idmap = pd.read_csv(idmap_path, sep="\t", header=0)
    gm = idmap[idmap["Source"] == "Gene Name"][["Preferred_Name", "Name"]]
    ensembl_to_symbol = dict(zip(gm["Preferred_Name"], gm["Name"]))
    with open(os.path.join(output_dir, "regulons.json")) as f:
        regulons = json.load(f)

    E = ext_expr.loc[:, ext_ids].dropna()
    ext_gene_set = set(E.index)

    def eigengene(M):

        '''

        First principal component of a regulon's genes across patients, oriented so that it rises with mean expression.

        Args: M (np.array), shape (n_genes, n_patients). Z-scored expression of the regulon's genes

        Returns: pc (np.array), shape (n_patients,). Eigengene, scaled to unit standard deviation

        '''

        M = M - M.mean(axis=1, keepdims=True)
        pc = np.linalg.svd(M, full_matrices=False)[2][0]
        if np.corrcoef(pc, M.mean(axis=0))[0, 1] < 0:
            pc = -pc
        return pc / pc.std()

    # Cox regression of survival on each regulon's eigengene in the validation cohort
    start, rows = time.time(), []
    for reg_id, genes in regulons.items():
        syms = [s for s in dict.fromkeys(ensembl_to_symbol.get(g, g) for g in genes) if s in ext_gene_set]
        if len(syms) < 3:
            continue
        df_cox = ext_surv.copy()
        df_cox["eigengene"] = eigengene(E.loc[syms].values)
        try:
            cph = CoxPHFitter().fit(df_cox, duration_col="OS_MONTHS", event_col="OS_STATUS")
            rows.append({"regulon": int(reg_id), "HR": cph.hazard_ratios_["eigengene"], "p_value": cph.summary.loc["eigengene", "p"]})
        except Exception:
            rows.append({"regulon": int(reg_id), "HR": np.nan, "p_value": np.nan})
    ext_cox = pd.DataFrame(rows).set_index("regulon")
    ext_cox.to_csv(os.path.join(output_dir, f"{val}_regulon_cox_results.csv"))
    print(f"Cox fitted for {len(ext_cox)} regulons in {(time.time()-start)/60:.1f} min")

    # Combine with the TCGA results
    tcga_cox  = pd.read_csv(os.path.join(output_dir, "regulon_cox_results.csv"), index_col=0)
    surv_tcga = set(tcga_cox.index[tcga_cox["p_value"] <= 0.05])
    coexp     = set(pd.read_csv(os.path.join(output_dir, f"{val}_coexp_validated_regulons.csv"), header=None)[0])
    surv_ext  = set(ext_cox.index[ext_cox["p_value"] <= 0.05])

    # Same direction: hazard ratio above 1 in both cohorts, or below 1 in both
    both = tcga_cox.join(ext_cox, rsuffix="_ext").dropna(subset=["HR", "HR_ext"])
    same_direction = set(both.index[(both["HR"] > 1) == (both["HR_ext"] > 1)])

    strict     = surv_tcga & coexp & surv_ext
    strict_dir = strict & same_direction
    pd.Series(sorted(strict_dir)).to_csv(os.path.join(output_dir, "final_disease_relevant_regulons_paper_rule.csv"), index=False, header=False)

    print(f"Survival-associated in TCGA:                         {len(surv_tcga)}")
    print(f"  and co-expression validated in {VAL_NAME}:          {len(surv_tcga & coexp)}")
    print(f"  and survival-associated in {VAL_NAME} (p <= 0.05):  {len(strict)}")
    print(f"  and with the same direction of effect:             {len(strict_dir)}")
    print("Paper: 1,083, of which 1,071 have survival replicated in Gravendeel")
else:
    print("No independent cohort with survival data: the paper-style rule cannot be applied")

GRAVENDEEL patients used: 264 | deaths: 239
Cox fitted for 3961 regulons in 4.1 min
Survival-associated in TCGA:                         1354
  and co-expression validated in GRAVENDEEL:          1154
  and survival-associated in GRAVENDEEL (p <= 0.05):  1148
  and with the same direction of effect:             1147
Paper: 1,083, of which 1,071 have survival replicated in Gravendeel


In [28]:
# Program-level survival association, as in the paper
# Program activity per patient: +1 if at least half of the program's regulons are overactive, -1 if at least half are
# underactive, otherwise 0 (the same rule as notebooks 02 and 05)

over  = pd.read_csv(os.path.join(output_dir, "overExpressedMembers.csv"),  index_col=0)
under = pd.read_csv(os.path.join(output_dir, "underExpressedMembers.csv"), index_col=0)
reg_activity = over.astype(int) - under.astype(int)
reg_activity.index = reg_activity.index.astype(int)

with open(os.path.join(output_dir, "transcriptional_programs.json")) as f:
    programs = json.load(f)

clin = pd.read_csv(os.path.join(output_dir, "clinical_data.csv"), index_col=0)[["OS_MONTHS", "OS_STATUS"]].dropna()
pts  = [p for p in reg_activity.columns if p in clin.index]

rows = []
for prog_id, regs in programs.items():
    regs = [int(r) for r in regs if int(r) in reg_activity.index]
    if not regs:
        continue
    net = reg_activity.loc[regs, pts].mean(axis=0)
    activity = (net >= 0.5).astype(int) - (net <= -0.5).astype(int)

    # A program that is neutral in (almost) every patient cannot be tested
    if (activity != 0).sum() < 5:
        rows.append({"program": int(prog_id), "n_regulons": len(regs), "HR": np.nan, "p_value": np.nan})
        continue
    df_cox = clin.loc[pts].copy()
    df_cox["activity"] = activity.values
    try:
        cph = CoxPHFitter().fit(df_cox, duration_col="OS_MONTHS", event_col="OS_STATUS")
        rows.append({"program": int(prog_id), "n_regulons": len(regs),
                     "HR": cph.hazard_ratios_["activity"], "p_value": cph.summary.loc["activity", "p"]})
    except Exception:
        rows.append({"program": int(prog_id), "n_regulons": len(regs), "HR": np.nan, "p_value": np.nan})

program_cox = pd.DataFrame(rows).set_index("program").sort_index()
program_cox.to_csv(os.path.join(output_dir, "program_cox_results.csv"))

sig = program_cox[program_cox["p_value"] <= 0.05]
print(f"Programs tested: {int(program_cox['p_value'].notna().sum())} of {len(program_cox)}")
print(f"Survival-associated (Cox p <= 0.05): {len(sig)} | high-risk (HR > 1): {int((sig['HR'] > 1).sum())} | low-risk (HR < 1): {int((sig['HR'] < 1).sum())}")
print("Paper: 58 of 179 programs | 51 high-risk | 7 low-risk")

Programs tested: 193 of 193
Survival-associated (Cox p <= 0.05): 35 | high-risk (HR > 1): 25 | low-risk (HR < 1): 10
Paper: 58 of 179 programs | 51 high-risk | 7 low-risk
